<a href="https://colab.research.google.com/github/mrishab/Semantic-Communication-Paper/blob/main/SemanticCommunication.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q gensim scikit-learn

In [ ]:
import re
import numpy as np
import pandas as pd

from gensim.models import Word2Vec
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.neighbors import NearestNeighbors
from sklearn.neural_network import MLPRegressor

np.random.seed(42)

# Simple Word2Vec Semantic Communication Demo

This is a simple end-to-end demo:
1. Split all sentences into words/tokens
2. Train a Word2Vec model to get word embeddings
3. Train a noise rebuilder on `(noisy_embedding -> clean_embedding)`
4. For any sentence: embed -> add noise -> rebuild -> inverse Word2Vec -> reconstructed sentence

In [ ]:
sentences = [
    "The quick brown fox jumps over the lazy dog.",
    "Data science is an interdisciplinary field.",
    "Artificial intelligence is transforming the world.",
    "I love working with python and pandas.",
    "Jupyter notebooks are great for prototyping.",
    "Natural language processing allows computers to understand text.",
    "Machine learning models require clean data for training."
]

sentences

In [ ]:
TOKEN_PATTERN = re.compile(r"\w+|[^\w\s]")

def tokenize(text):
    return TOKEN_PATTERN.findall(text.lower())

def detokenize(tokens):
    if not tokens:
        return ""

    out = tokens[0]
    for tok in tokens[1:]:
        if tok in ".,!?;:":
            out += tok
        else:
            out += " " + tok
    return out

tokenized_sentences = [tokenize(s) for s in sentences]
tokenized_sentences

In [ ]:
# Train a small Word2Vec model on the demo corpus
w2v = Word2Vec(
    sentences=tokenized_sentences,
    vector_size=64,
    window=3,
    min_count=1,
    workers=1,
    sg=1,
    epochs=500,
    seed=42,
)

vocabulary = list(w2v.wv.index_to_key)
clean_vectors = np.vstack([w2v.wv[word] for word in vocabulary]).astype(np.float32)

print(f"Vocabulary size: {len(vocabulary)}")
print(f"Embedding shape: {clean_vectors.shape}")

In [ ]:
def make_denoising_data(clean_vecs, noise_std=0.10, repeats=250):
    """Create (noisy -> clean) training pairs from vocabulary embeddings."""
    vocab_size, dim = clean_vecs.shape

    noise = np.random.normal(0, noise_std, size=(repeats, vocab_size, dim)).astype(np.float32)
    noisy = clean_vecs[None, :, :] + noise

    X_train = noisy.reshape(repeats * vocab_size, dim)
    y_train = np.repeat(clean_vecs[None, :, :], repeats, axis=0).reshape(repeats * vocab_size, dim)
    return X_train, y_train

train_noise_std = 0.10
X_train, y_train = make_denoising_data(clean_vectors, noise_std=train_noise_std, repeats=250)

rebuilder = MLPRegressor(
    hidden_layer_sizes=(128, 128),
    activation="relu",
    solver="adam",
    max_iter=400,
    early_stopping=True,
    random_state=42,
)
rebuilder.fit(X_train, y_train)

print(f"Rebuilder trained on {len(X_train)} noisy samples.")

In [ ]:
# Inverse Word2Vec: nearest vocabulary vector lookup
inverse_index = NearestNeighbors(n_neighbors=1, metric="cosine")
inverse_index.fit(clean_vectors)

print("Inverse Word2Vec index ready.")

In [ ]:
def sentence_to_vectors(sentence):
    tokens = tokenize(sentence)
    vectors = []
    known_mask = []

    for tok in tokens:
        if tok in w2v.wv:
            vectors.append(w2v.wv[tok])
            known_mask.append(True)
        else:
            vectors.append(np.zeros(w2v.vector_size, dtype=np.float32))
            known_mask.append(False)

    return tokens, np.vstack(vectors).astype(np.float32), np.array(known_mask, dtype=bool)

def inverse_word2vec(vectors, known_mask, original_tokens):
    distances, indices = inverse_index.kneighbors(vectors)

    recovered = []
    for i in range(len(original_tokens)):
        if known_mask[i]:
            recovered.append(vocabulary[indices[i, 0]])
        else:
            # Keep unknown tokens unchanged
            recovered.append(original_tokens[i])

    return recovered

def reconstruct_sentence(sentence, noise_std=0.10):
    # 1) sentence -> word embeddings
    tokens, clean_vecs, known_mask = sentence_to_vectors(sentence)

    # 2) add channel noise
    noise = np.random.normal(0, noise_std, size=clean_vecs.shape).astype(np.float32)
    noisy_vecs = clean_vecs + noise

    # 3) denoise with rebuilder model
    rebuilt_vecs = rebuilder.predict(noisy_vecs).astype(np.float32)

    # 4) inverse Word2Vec (nearest word vector)
    recovered_tokens = inverse_word2vec(rebuilt_vecs, known_mask, tokens)

    # 5) reconstruct sentence
    reconstructed_sentence = detokenize(recovered_tokens)

    # token-level debug table
    rows = []
    for i, tok in enumerate(tokens):
        if known_mask[i]:
            noisy_sim = float(cosine_similarity(clean_vecs[i:i+1], noisy_vecs[i:i+1])[0, 0])
            rebuilt_sim = float(cosine_similarity(clean_vecs[i:i+1], rebuilt_vecs[i:i+1])[0, 0])
        else:
            noisy_sim = np.nan
            rebuilt_sim = np.nan

        rows.append({
            "Token": tok,
            "Recovered": recovered_tokens[i],
            "Match": tok == recovered_tokens[i],
            "Noisy Sim": round(noisy_sim, 4) if not np.isnan(noisy_sim) else np.nan,
            "Rebuilt Sim": round(rebuilt_sim, 4) if not np.isnan(rebuilt_sim) else np.nan,
        })

    details = pd.DataFrame(rows)
    return {
        "original": sentence,
        "reconstructed": reconstructed_sentence,
        "details": details,
    }

In [ ]:
demo_sentence = "The quick brown fox jumps over the lazy dog."
result = reconstruct_sentence(demo_sentence, noise_std=0.15)

print("Original:     ", result["original"])
print("Reconstructed:", result["reconstructed"])
display(result["details"])

In [ ]:
custom_sentence = "Machine learning with python is great."
custom_result = reconstruct_sentence(custom_sentence, noise_std=0.15)

print("Original:     ", custom_result["original"])
print("Reconstructed:", custom_result["reconstructed"])
display(custom_result["details"])

In [ ]:
def evaluate_sentences(sentence_list, noise_std=0.15):
    rows = []
    for s in sentence_list:
        out = reconstruct_sentence(s, noise_std=noise_std)
        acc = float(out["details"]["Match"].mean())
        rows.append({
            "Original": s,
            "Reconstructed": out["reconstructed"],
            "Word Accuracy": f"{acc:.1%}",
        })
    return pd.DataFrame(rows)

display(evaluate_sentences(sentences, noise_std=0.15))